# Lab 12: Training at scale

How does a network learn its weights, and how is that done on data too big for one machine? Tonight we start with one weight and gradient descent. We let the computer take derivatives (automatic differentiation). We add K classes, minibatches, and many GPUs (simulated on your CPU). We end with the same network in PyTorch.

Only one optimizer appears in this course: plain stochastic gradient descent (SGD). The update is always $w \leftarrow w - \eta \times \text{gradient}$, where $\eta$ ("eta") is the learning rate. In code the learning rate is called `lr`.

1. Gradient descent on one weight
2. Automatic differentiation: the computational graph, forward and reverse mode, and JAX
3. K classes: softmax output, cross-entropy, its gradient, the stable version
4. Minibatches: SGD and the memory budget
5. Many GPUs: data-parallel SGD as map-reduce (and a note on power)
6. The same network in PyTorch, diagnosing training, weight decay and early stopping

Then the lab exercises (6), the exit problem, and the work at home. Sections 3 to 6 contain the computing corner on tensors: axes, dtypes, NumPy against PyTorch, devices, and memory in bytes.


## Route

By the end of this lab you should be able to:

1. Compute one gradient-descent update and a chain-rule derivative by hand, and explain how reverse-mode automatic differentiation gets all the derivatives of a loss in one backward pass.
2. Write the softmax cross-entropy gradient $p - y$ with arrays along the right axis, and explain why averaging shard gradients gives the same step as one big batch.
3. Train a small network in PyTorch with plain SGD, read the training and validation curves, restore the best epoch, and test once.

Required: Sections 1 to 6, the exit problem, and the five graded exercises (12.1 to 12.4 in class, 12.5 at home). The recorded Hopper tables (all-reduce times and measured power) are part of Section 5.

Due: 7 December, the last day of classes. Submit the notebook on Canvas.

Run All stops at the first empty exercise check; that is expected. Fill the exercises, then Restart and Run All once. For the test set, follow the order in Exercise 12.5: run the final-evaluation cell once after that, and do not restart again before you submit.


### Online labs for this notebook

- [Neural Net Playground](https://vsokolov.org/labs/neural-net-lab.html): train a small network and watch the loss fall and the boundary form (Sections 1 and 2). It starts on the Adam optimizer: set it to plain SGD before you train.
- [Next-Token Lab](https://vsokolov.org/labs/next-token-lab.html): a language model's choice of the next word is a softmax over the vocabulary (Section 3).


In [ ]:
import os

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # quiet JAX start-up messages
import time
from hashlib import sha256
from urllib.request import urlretrieve

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import copy
import torch
from jax import grad, jit, random
from torch import nn

os.makedirs("data", exist_ok=True)
os.makedirs("fig", exist_ok=True)
BASE = "https://vsokolov.org/courses/files/600"
for name in ("data/circle.csv",):
    if not os.path.exists(name):
        urlretrieve(f"{BASE}/{name}", name)

def check(name, value, expected, ndigits=6):
    if value is None:
        raise AssertionError(f"replace None for {name}")
    s = f"{round(float(value), ndigits):.{ndigits}f}"
    got = sha256(f"aii600-lab12|{name}|{s}".encode()).hexdigest()[:16]
    assert got == expected, f"{name} is not right"
    print("ok ", name)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

np.set_printoptions(precision=4, suppress=True)


## 1. Gradient descent on one weight

- Logistic regression with one weight $w$ and no intercept: the probability of a 1 is $p_i = \sigma(w x_i)$, where $\sigma(z) = 1/(1+e^{-z})$.
- The loss is the average log loss over the $n$ examples: $L(w) = -\frac{1}{n}\sum_i \big[y_i \log p_i + (1-y_i)\log(1-p_i)\big]$. In code we compute the same number from the score $a_i = w x_i$ as $\log(1 + e^{a_i}) - y_i a_i$, which stays finite for large scores (Section 3 explains why).
- The derivative of the loss with respect to $w$ is $\frac{dL}{dw} = \frac{1}{n}\sum_i (p_i - y_i)\, x_i$. The factor $p_i - y_i$ is the miss: probability minus label.
- Gradient descent moves $w$ a small step against the derivative: $w \leftarrow w - \eta\, \frac{dL}{dw}$.
- Below, the data are simulated: 30 values of $x$ from a normal distribution and labels drawn with $p = \sigma(1.5\,x)$. The true weight is 1.5.


In [ ]:
rng = np.random.default_rng(1)
x1 = rng.normal(size=30)
y1 = (rng.random(30) < sigmoid(1.5 * x1)).astype(float)

def loss1(w):
    a = w * x1                                  # the score (logit) of each example
    return np.mean(np.logaddexp(0, a) - y1 * a)  # the log loss, written with logits so it never takes log(0)

def dloss1(w):
    return np.mean((sigmoid(w * x1) - y1) * x1)

# One update by hand: start at w = 0 with learning rate 1.
w_start, lr = 0.0, 1.0
g_start = dloss1(w_start)
w_next = w_start - lr * g_start
print(f"loss at w = 0:    {loss1(w_start):.4f}")
print(f"derivative at 0:  {g_start:.4f}")
print(f"new w:            {w_next:.4f}")
print(f"loss at new w:    {loss1(w_next):.4f}")


**Reading.** At $w = 0$ every probability is 0.5, the loss is 0.6931 (that is $\log 2$), and the derivative is -0.2287. The derivative is negative, so the step $-\eta \times$ derivative is positive and $w$ moves up to 0.2287. The loss falls to 0.6453.


In [ ]:
w, path = 0.0, []
for step in range(51):
    path.append((w, loss1(w)))
    w = w - lr * dloss1(w)
path = np.array(path)
ws = np.linspace(-0.5, 3.5, 200)
w_hat = ws[np.argmin([loss1(v) for v in ws])]
loss_best = loss1(w_hat)
print(f"after 50 steps: w = {path[-1, 0]:.3f}, loss = {path[-1, 1]:.4f}")
print(f"best w on a grid: {w_hat:.3f}, loss = {loss_best:.4f}")

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(ws, [loss1(v) for v in ws], color="C0", label="loss curve")
ax.plot(path[:, 0], path[:, 1], "o--", color="C3", ms=3, label="gradient descent path")
ax.set_xlabel("weight w")
ax.set_ylabel("average log loss")
ax.legend(frameon=False)
fig.tight_layout()


**Reading.** The red dots start at $w = 0$ and slide down the blue curve. The first steps are big and later steps are tiny, because the derivative shrinks as the curve flattens. After 50 steps $w$ = 2.377, close to the best value on the grid (2.676). It is not the true 1.5: with 30 simulated examples the fitted weight is noisy.


**Predict.** The next cell uses a learning rate 100 times larger (`lr = 100`). Will the loss fall smoothly and faster, or do something else? Write your guess, then run it.


In [ ]:
w, losses = 0.0, []
for step in range(8):
    losses.append(loss1(w))
    w = w - 100 * dloss1(w)
print(np.round(losses, 3))


**Reading.** The loss does not fall smoothly. The first step jumps to $w$ = 22.9, far past the bottom of the curve, and the loss goes **up**, to 1.705. Then it falls back (1.202, 0.737, 0.479). It recovers here only because this problem is mild: the sigmoid flattens far from the bottom, so the gradient is small and the next steps are not wild. A step that is too large overshoots and wastes steps, and with a larger rate (or a steeper loss) it does not recover. The exploding run in Section 6 shows a rate that fails. A learning rate has to be small enough for the curve's steepness.


**Your turn.** Find a learning rate that gets the loss after 50 steps within 0.001 of `loss_best` (the best loss on the grid, printed above). Change `lr_try` and run. The cell prints whether you reached the target.


In [ ]:
lr_try = 0.01  # change this
w = 0.0
for step in range(50):
    w = w - lr_try * dloss1(w)
print(f"loss after 50 steps: {loss1(w):.4f}, target: {loss_best + 0.001:.4f}")
print("target reached:", loss1(w) <= loss_best + 0.001)


**Online lab.** Open the [Neural Net Playground](https://vsokolov.org/labs/neural-net-lab.html) next to this section. Try this: first set the optimizer to plain SGD (it starts on Adam), so that its learning rate means the same as our $\eta$. Then press train with a small learning rate, then raise the learning rate step by step and note when the loss stops falling and starts to jump.


**Check yourself.**

1. At $w = 0$ the derivative is negative. In which direction does gradient descent move $w$, and why?
2. The loss curve is a bowl. What happens to the step size as $w$ approaches the bottom, if $\eta$ stays fixed?

<details><summary>Answer</summary>

1. Up (to larger $w$). The update subtracts $\eta$ times the derivative, and minus a negative number is positive.
2. The steps shrink, because the derivative gets closer to 0 near the bottom. That is why the red dots bunch up.

</details>


In [ ]:
# Inputs for Exercise 12.1
x_ex = np.array([1.0, -2.0, 0.5, 3.0])
y_ex = np.array([1.0, 0.0, 0.0, 1.0])
w_start_ex, eta_ex = 0.5, 0.4


### Exercise 12.1 (In class)

**Task:** do one gradient-descent update by hand for the one-weight logistic regression of Section 1, on the four examples `x_ex` and `y_ex`, starting at `w_start_ex` with learning rate `eta_ex`.

- Inputs: `x_ex`, `y_ex` (arrays of length 4), `w_start_ex`, `eta_ex`.
- Define `grad_ex` (a number): the derivative of the average log loss at `w_start_ex`, which is the mean of $(p_i - y_i)\,x_i$ with $p_i = \sigma(w x_i)$.
- Define `w_new` (a number): `w_start_ex - eta_ex * grad_ex`.


In [ ]:
# Exercise 12.1: define grad_ex, w_new
grad_ex = None
w_new = None


In [ ]:
check("grad_ex", grad_ex, "f833f63447cb7119")
check("w_new", w_new, "b58f77d72f1f9234")
print("ok  exercise")


## 2. Automatic differentiation

Gradient descent needs derivatives. There are three ways to get them.

1. **By hand.** Slow and easy to get wrong.
2. **Finite differences.** Use $f'(x) \approx \frac{f(x+h) - f(x)}{h}$ for a small $h$. It is approximate, and it breaks down if $h$ is too small (see below).
3. **Automatic differentiation (AD).** The computer applies the chain rule to your code. The answer is accurate up to floating-point arithmetic (and follows a convention at points where a function has no derivative), at the point where you ask. You do not get a formula, only the value.

The example is a one-neuron model, $f = \sigma(w x + b)$, written as three lines of code:

- `v1 = w * x`
- `v2 = v1 + b`
- `v3 = sigmoid(v2)`

This is a computational graph: each variable is a node, each arrow is one operation. The forward pass computes the values. The backward pass pushes derivatives back. We draw it below, after both passes, with the numbers of this notebook.

The book's example uses $b = 5$. There the sigmoid is saturated ($f = 0.99998$) and every derivative is about 0.00005, which hides the arithmetic. We use $x = 2$, $w = 3$, $b = -5$ instead. Every derivative below is a derivative of the output $f$ (not of a later training loss).

We need one fact: the derivative of the sigmoid is $\sigma'(z) = \sigma(z)\,(1 - \sigma(z))$.


In [ ]:
xv, wv, bv = 2.0, 3.0, -5.0
v1 = wv * xv
v2 = v1 + bv
v3 = sigmoid(v2)
local = v3 * (1 - v3)          # d v3 / d v2
print(f"v1 = {v1}, v2 = {v2}, v3 = f = {v3:.3f}")
print(f"sigmoid slope at v2: {local:.4f}")


**Reading.** $v_1 = 6$, $v_2 = 1$, and $f = \sigma(1) = 0.731$. The sigmoid slope at $v_2 = 1$ is 0.1966. We reuse this number in both modes.


### Forward mode (dual numbers)

Carry a derivative next to every value, from the inputs to the output. The pair (value, derivative) is a **dual number**. We ask for the derivative with respect to $x$, so the starting pair for $x$ is $(2, 1)$ and for $w$ and $b$ it is $(w, 0)$ and $(b, 0)$.

| Value | Derivative with respect to $x$ |
|---|---|
| `v1 = w * x = 6` | `dv1 = w * 1 + 0 * x = 3` |
| `v2 = v1 + b = 1` | `dv2 = dv1 = 3` |
| `v3 = sigmoid(v2) = 0.731` | `dv3 = dv2 * v3 * (1 - v3) = 0.590` |

One forward pass gives the derivative with respect to **one** input. For $w$ we would run it again with the starting pairs $(x, 0)$, $(w, 1)$, $(b, 0)$.


In [ ]:
# Forward mode with dual numbers: each variable is a pair (value, derivative).
def forward(x, w, b, seed):
    # seed says which input has derivative 1: "x", "w" or "b"
    X = (x, 1.0 if seed == "x" else 0.0)
    W = (w, 1.0 if seed == "w" else 0.0)
    B = (b, 1.0 if seed == "b" else 0.0)
    V1 = (W[0] * X[0], W[0] * X[1] + W[1] * X[0])   # product rule
    V2 = (V1[0] + B[0], V1[1] + B[1])               # sum rule
    s = sigmoid(V2[0])
    V3 = (s, V2[1] * s * (1 - s))                   # chain rule
    return V3

for seed in ("x", "w", "b"):
    value, deriv = forward(xv, wv, bv, seed)
    print(f"forward pass seeded on {seed}: f = {value:.3f}, df/d{seed} = {deriv:.3f}")


**Reading.** Three forward passes, one per input: $df/dx$ = 0.590, $df/dw$ = 0.393, $df/db$ = 0.197. Each pass costs about as much as computing $f$ itself. With three inputs that is fine. With a million weights it would be a million passes.


### Reverse mode (backpropagation)

Start at the output, with derivative 1 for $f$ itself, and push the derivative **back** through the graph. At each node multiply by the local derivative.

| Step | Backward pass |
|---|---|
| 1 | `df/dv3 = 1` |
| 2 | `df/dv2 = df/dv3 * v3 * (1 - v3) = 0.1966` |
| 3 | `df/dv1 = df/dv2 * 1 = 0.1966`, and `df/db = df/dv2 * 1 = 0.1966` |
| 4 | `df/dx = df/dv1 * w = 0.590`, and `df/dw = df/dv1 * x = 0.393` |

One backward pass gives the derivative with respect to **every** input at once. The diagram after the next code cell shows the two directions on the graph.

- **Forward mode** carries derivatives with the computation. It is cheap when there are few inputs and many outputs.
- **Reverse mode** carries them against the computation. It is cheap when there are many inputs and one output.
- A network has millions of inputs (the weights) and one output (the loss). So networks use reverse mode. **Backpropagation is reverse-mode automatic differentiation.**


In [ ]:
# Reverse mode by hand: one backward pass.
df_dv3 = 1.0
df_dv2 = df_dv3 * v3 * (1 - v3)
df_dv1 = df_dv2 * 1.0
rev_db = df_dv2 * 1.0
rev_dx = df_dv1 * wv
rev_dw = df_dv1 * xv
print(f"reverse mode: df/dx = {rev_dx:.3f}, df/dw = {rev_dw:.3f}, df/db = {rev_db:.3f}")

# Finite differences, for comparison (float64).
def f_all(x, w, b):
    return sigmoid(w * x + b)

for h in (1e-1, 1e-3, 1e-6, 1e-10, 1e-14):
    approx = (f_all(xv, wv + h, bv) - f_all(xv, wv, bv)) / h
    print(f"h = {h:.0e}: finite difference for df/dw = {approx:.8f}, error = {abs(approx - rev_dw):.1e}")


**Reading.** Reverse mode gives the same numbers as forward mode: 0.590, 0.393, 0.197. The finite difference is the bad deal: with $h = 0.1$ the error is 1.9e-02 (the step is too big), and with $h = 10^{-14}$ the error grows to 6.5e-03 (the two function values are almost equal, so rounding errors dominate). The best $h$ is in the middle, and it is different for every function. AD has no $h$.


In [ ]:
# One correct picture of both passes, with this notebook's numbers. All derivatives are derivatives of f.
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.set_xlim(0, 10); ax.set_ylim(0, 5); ax.axis("off")
nodes = {"x": (0.8, 4.0), "w": (0.8, 2.6), "v1": (3.2, 3.3), "b": (3.2, 0.9), "v2": (5.6, 2.2), "f": (8.2, 2.2)}
fwd = {"x": xv, "w": wv, "b": bv, "v1": v1, "v2": v2, "f": v3}
bwd = {"x": rev_dx, "w": rev_dw, "b": rev_db, "v1": df_dv1, "v2": df_dv2, "f": df_dv3}
names = {"x": "x", "w": "w", "b": "b", "v1": "v1 = w*x", "v2": "v2 = v1 + b", "f": "f = sigmoid(v2)"}
for k, (px, py) in nodes.items():
    ax.text(px, py, f"{names[k]}\nvalue {fwd[k]:.3f}\ndf/d{k} = {bwd[k]:.3f}" if k != "f" else f"{names[k]}\nvalue {fwd[k]:.3f}\ndf/df = 1",
            ha="center", va="center", fontsize=8, bbox=dict(boxstyle="round", fc="white", ec="0.3"))
edges = [("x", "v1"), ("w", "v1"), ("v1", "v2"), ("b", "v2"), ("v2", "f")]
for a_, b_ in edges:
    ax.annotate("", xy=nodes[b_], xytext=nodes[a_], arrowprops=dict(arrowstyle="->", color="C0", lw=1.5, shrinkA=38, shrinkB=38, connectionstyle="arc3,rad=0.12"))
    ax.annotate("", xy=nodes[a_], xytext=nodes[b_], arrowprops=dict(arrowstyle="->", color="C3", lw=1.5, ls="--", shrinkA=38, shrinkB=38, connectionstyle="arc3,rad=0.12"))
ax.text(5, 4.7, "solid blue arrows: forward pass (values).  dashed red arrows: backward pass (derivatives of f).", ha="center", fontsize=8)
fig.tight_layout()


**Reading.** Blue solid arrows carry the values forward: $v_1 = 6$, $v_2 = 1$, $f = 0.731$. Red dashed arrows carry the derivatives of $f$ backward, starting from $df/df = 1$. Each box shows its value and the derivative of $f$ with respect to it: $df/dv_2 = df/dv_1 = df/db = 0.197$, $df/dx = 0.590$, $df/dw = 0.393$. The tables above are the reference; the picture shows the same numbers on the graph.


### JAX: `grad`, `argnums`, `jit`

JAX is a library that does reverse-mode AD on ordinary Python and NumPy-style code.

- `jax.grad(f)` returns a **new function** that computes the derivative of `f` with respect to its first argument.
- `argnums=(0, 1, 2)` asks for the derivative with respect to several arguments at once.
- `jax.jit` compiles a function, so it runs faster after the first call.
- JAX works in float32 by default, so the numbers agree with ours to about 7 digits.


In [ ]:
def f_jax(x, w, b):
    return 1 / (1 + jnp.exp(-(w * x + b)))

print("f      =", float(f_jax(xv, wv, bv)))
print("df/dx  =", float(grad(f_jax)(xv, wv, bv)))
dx, dw, db = grad(f_jax, argnums=(0, 1, 2))(xv, wv, bv)
print(f"all three at once: df/dx = {float(dx):.3f}, df/dw = {float(dw):.3f}, df/db = {float(db):.3f}")
fast = jit(grad(f_jax, argnums=(0, 1, 2)))
print("jit version:", [round(float(a), 3) for a in fast(xv, wv, bv)])


**Reading.** `grad` returns 0.590, 0.393, 0.197, the same as the hand tables, from one call. We never wrote a derivative formula. That is the point.


### The book's circle network in JAX

- Data: 200 points in the plane with a 0/1 label (`data/circle.csv`, columns `label`, `x1`, `x2`). Label 0 is the center cloud and label 1 is the outer ring, as in Lab 10. A straight line cannot separate them.
- Network 2-4-1: two inputs, four hidden units with a sigmoid, one output unit with a sigmoid. Lab 10 used ReLU hidden units. Here we use sigmoid hidden units because this is the book's example, and the sigmoid has the simple derivative $\sigma(1-\sigma)$ used in the hand calculation above. It is a different network from Lab 10's, with the same number of parameters (17).
- Loss: the negative log-likelihood, summed over the 200 points.
- Training: plain gradient descent, 1,500 steps, learning rate 0.02. At each step `grad` gives the gradient for all four weight arrays (`w1`, `b1`, `w2`, `b2`) and `jit` compiles the step.
- JAX has no hidden random state. Every random draw needs an explicit **key**. The book makes one key and uses it four times.


In [ ]:
circle = pd.read_csv("data/circle.csv")
xc = circle[["x1", "x2"]].to_numpy()
yc = circle["label"].to_numpy().astype(float)
radius = np.hypot(xc[:, 0], xc[:, 1])
print("data:", xc.shape, yc.shape, "share of ones:", yc.mean())
print(f"label 0: radius from {radius[yc == 0].min():.2f} to {radius[yc == 0].max():.2f}")
print(f"label 1: radius from {radius[yc == 1].min():.2f} to {radius[yc == 1].max():.2f}")


**Reading.** Half of the 200 points have label 1. Label 0 (distance from the origin 0.01 to 3.95) is the center cloud. Label 1 (8.00 to 9.99) is the outer ring.


In [ ]:
k = random.PRNGKey(0)                       # the book: one key for all four arrays
w1 = 0.1 * random.normal(k, (2, 4))
b1 = 0.01 * random.normal(k, (4,))
w2 = 0.1 * random.normal(k, (4, 1))
b2 = 0.01 * random.normal(k, (1,))
print("b1 / 0.01      :", np.array(b1 / 0.01))
print("w1[0, :] / 0.1:  ", np.array(w1.ravel()[:4] / 0.1))
print("w2 / 0.1       :", np.array(w2.ravel() / 0.1))
print("b2 / 0.01      :", np.array(b2 / 0.01))


**Reading.** The same key gives the same random numbers. The same four numbers (1.6226, 2.0253, ...) appear in `b1`, in the first row of `w1`, and in `w2` (and `b2` is the first of them). Only the scale (0.01 or 0.1) differs. So these arrays are not independent: they are scaled copies of one list of four numbers. That is a bug in how the book draws its starting weights, even though the network still trains.


In [ ]:
def logit(x, w1, b1, w2, b2):
    z = 1 / (1 + jnp.exp(-(jnp.dot(x, w1) + b1)))      # sigmoid hidden units
    return (jnp.dot(z, w2) + b2)[:, 0]                  # the output score

def predict(x, w1, b1, w2, b2):
    return 1 / (1 + jnp.exp(-logit(x, w1, b1, w2, b2)))

def nll(x, y, w1, b1, w2, b2):
    a = logit(x, w1, b1, w2, b2)
    return jnp.sum(jax.nn.softplus(a) - y * a)           # negative log-likelihood from the score (stable)

@jit
def gd_step(x, y, w1, b1, w2, b2, lr):
    g = grad(nll, argnums=(2, 3, 4, 5))(x, y, w1, b1, w2, b2)
    return w1 - lr * g[0], b1 - lr * g[1], w2 - lr * g[2], b2 - lr * g[3]

def accuracy(x, y, *params):
    return float(jnp.mean((predict(x, *params) > 0.5) == y))

def train_circle(params, steps=1500, lr=0.02):
    # JAX runs asynchronously, so block on the result before stopping each timer.
    t0 = time.time()
    params = gd_step(xc, yc, *params, lr)
    jax.block_until_ready(params)
    first = time.time() - t0                  # includes compiling the step the first time
    t0 = time.time()
    for i in range(steps - 1):
        params = gd_step(xc, yc, *params, lr)
    jax.block_until_ready(params)
    return params, first, time.time() - t0

params_book, first_book, rest_book = train_circle((w1, b1, w2, b2))
print(f"book init:  accuracy {accuracy(xc, yc, *params_book):.3f}, loss {float(nll(xc, yc, *params_book)):.2f}")
print(f"            first step (with compile) {first_book:.3f} s, other 1,499 steps {rest_book:.3f} s")

# The fix: one key per array.
k1, k2, k3, k4 = random.split(random.PRNGKey(0), 4)
init_split = (0.1 * random.normal(k1, (2, 4)), 0.01 * random.normal(k2, (4,)),
              0.1 * random.normal(k3, (4, 1)), 0.01 * random.normal(k4, (1,)))
print("b1 / 0.01 with split keys:", np.array(init_split[1] / 0.01))
params_split, first_split, rest_split = train_circle(init_split)
print(f"split keys: accuracy {accuracy(xc, yc, *params_split):.3f}, loss {float(nll(xc, yc, *params_split)):.2f}")
print(f"            first step (already compiled) {first_split:.3f} s, other 1,499 steps {rest_split:.3f} s")


**Reading.** Both runs reach accuracy 1.000 (all 200 points right). With split keys the final loss is 0.45, against 0.54 with the book's reused key. The timing lines separate the one-time compile of `jit` (the first step of the first run) from the steady-state steps; we wait for the result with `block_until_ready` before stopping the timer, because JAX returns before the work is done. Times differ on your machine. The reused key did not break this small example, but it is a trap: in a bigger network, correlated starting weights can make units behave alike. `jax.random.split` is the habit to keep: one key per random array.


In [ ]:
lo, hi = xc.min(axis=0) - 1, xc.max(axis=0) + 1          # the mesh covers the data range plus a margin
gx, gy = np.meshgrid(np.linspace(lo[0], hi[0], 200), np.linspace(lo[1], hi[1], 200))
grid = np.c_[gx.ravel(), gy.ravel()]
prob = np.array(predict(grid, *params_split)).reshape(gx.shape)
fig, ax = plt.subplots(figsize=(4.4, 4.0))
ax.contourf(gx, gy, prob, levels=[0, 0.5, 1], colors=["#fdd", "#ddf"], alpha=0.8)
ax.contour(gx, gy, prob, levels=[0.5], colors="k", linewidths=1.5)
ax.scatter(xc[yc == 1, 0], xc[yc == 1, 1], s=10, color="C0", label="label 1")
ax.scatter(xc[yc == 0, 0], xc[yc == 0, 1], s=10, marker="x", color="C3", label="label 0")
ax.set_xlabel("first input")
ax.set_ylabel("second input")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2)
fig.tight_layout()


**Reading.** The mesh covers all the points. The black curve is where the network gives probability 0.5. The blue region is where the network says "label 1" (the outer ring) and the red region is where it says "label 0" (the center). Four sigmoid units combine into a closed shape around the center cloud. Each hidden unit draws one soft line, and the output unit combines the four lines. No single line could do this.


### The backward pass as matrix products

`grad` hides the arithmetic. Here it is for a batch of $n$ points. The forward pass is two matrix products:

- `A1 = X @ W1 + b1` has shape (n, 4), and `Z = sigmoid(A1)`
- `A2 = Z @ W2 + b2` has shape (n, 1), and `P = sigmoid(A2)`

The backward pass goes the other way. Each step multiplies by the **transposed** weights:

- `dA2 = P - y` has shape (n, 1), with `y` as a column. This gradient is for the **sum** loss (not divided by n), unlike Section 1 and Exercise 12.1. (For a sigmoid output with the log loss, the derivative with respect to the score is probability minus label. Section 3 returns to this.)
- `dW2 = Z.T @ dA2` has shape (4, 1), and `db2` is the column sum of `dA2`
- `dZ = dA2 @ W2.T` has shape (n, 4)
- `dA1 = dZ * Z * (1 - Z)` has shape (n, 4)
- `dW1 = X.T @ dA1` has shape (2, 4), and `db1` is the column sum of `dA1`

Each layer needs two matrix products going backward (one for the weights, one for the signal passed on) and only one going forward. So a training step costs about **twice** the forward pass for the matrix products, plus the forward pass itself: about three times a forward pass in total. Let us check these formulas against `grad`.


In [ ]:
W1, B1, W2, B2 = [np.array(a, dtype=float) for a in params_split]
X64 = xc.astype(float)
A1 = X64 @ W1 + B1;  Z = sigmoid(A1)
A2 = Z @ W2 + B2;    P = sigmoid(A2)[:, 0]
y_col = yc[:, None]              # y as a column, shape (200, 1)
dA2 = P[:, None] - y_col
dW2 = Z.T @ dA2;     dB2 = dA2.sum(axis=0)
dZ = dA2 @ W2.T
dA1 = dZ * Z * (1 - Z)
dW1 = X64.T @ dA1;   dB1 = dA1.sum(axis=0)
by_hand = [dW1, dB1, dW2, dB2]
by_jax = grad(nll, argnums=(2, 3, 4, 5))(xc, yc, *params_split)
print("shapes by hand:", [a.shape for a in by_hand])
for nm, a, b in zip(("w1", "b1", "w2", "b2"), by_hand, by_jax):
    print(f"{nm}: largest gradient entry {np.abs(a).max():.4f}, largest difference from jax.grad {np.abs(a - np.array(b)).max():.1e}")


**Reading.** The shapes are (2, 4), (4,), (4, 1), (1,), the same as the weights. The hand-written backward pass agrees with `jax.grad` to about 1.5e-05 (JAX uses float32, our arrays use float64). The gradients are small because the network already classifies nearly every point correctly, so every $p - y$ is small. (The two classes can be separated perfectly, so the loss has no finite minimum: it is still slowly falling.) Backpropagation is nothing more than these lines.


**Check yourself.**

1. A model has 10 million weights and one scalar loss. Which mode, forward or reverse, gets all 10 million derivatives in one pass?
2. The same key is used for `w1` and `b1`. What do you see in the numbers?
3. What does `jax.grad(f)` return: a number or a function?

<details><summary>Answer</summary>

1. Reverse mode. One backward pass from the loss gives the derivative with respect to every weight. Forward mode would need 10 million passes.
2. `b1` is a scaled copy of the first four entries of `w1`. Use `jax.random.split` to give each array its own key.
3. A function. Call it on the inputs to get the derivative.

</details>


In [ ]:
# Inputs for Exercise 12.2
x_r, w_r, b_r = 1.5, 2.0, -4.0


### Exercise 12.2 (In class)

**Task:** by reverse mode, find the derivatives of $f = \sigma(w x + b)$ at $x$ = `x_r`, $w$ = `w_r`, $b$ = `b_r`. Run the forward pass (`v1 = w*x`, `v2 = v1 + b`, `v3 = sigmoid(v2)`), then one backward pass starting from `df/dv3 = 1`.

- Inputs: `x_r`, `w_r`, `b_r` (numbers).
- Define `df_dw` and `df_dx` (numbers).


In [ ]:
# Exercise 12.2: define df_dw, df_dx
df_dw = None
df_dx = None


In [ ]:
check("df_dw", df_dw, "687e1e96b5da6d7f")
check("df_dx", df_dx, "29792ff1b417cc0c")
print("ok  exercise")


## 3. K classes: softmax, cross-entropy, and the gradient $p - y$

So far the output was one number (a probability of "1"). For $K$ classes the output layer produces $K$ raw numbers, the **scores** $s_1, \dots, s_K$ (also called logits). Three steps turn them into a loss.

1. **Softmax** turns scores into probabilities: $p_j = e^{s_j} / \sum_k e^{s_k}$. All $p_j$ are positive and they add to 1.
2. The label is a **one-hot** vector $y$: $y_j = 1$ for the true class and 0 for the others.
3. The **cross-entropy** loss is minus the log of the probability of the true class: $L = -\sum_j y_j \log p_j$.

This is the same log loss as in the regularization and priors topic, now for $K$ classes.

### The gradient, on paper, for $K = 3$

If the true class is $c$, then $L = -\log p_c = -s_c + \log\left(e^{s_1} + e^{s_2} + e^{s_3}\right)$. Take the derivative with respect to the score $s_j$:

- The first term $-s_c$ gives $-1$ if $j = c$ and 0 otherwise. That is $-y_j$.
- The second term gives $e^{s_j} / (e^{s_1} + e^{s_2} + e^{s_3}) = p_j$.

So $\dfrac{\partial L}{\partial s_j} = p_j - y_j$. **The gradient with respect to the scores is the probabilities minus the one-hot label.** This one line is where backpropagation starts for every classifier. For a batch of $n$ examples and the **mean** loss, the gradient with respect to the score matrix is $(P - Y)/n$, where $P$ and $Y$ have one row per example. It matches the sigmoid case in Section 2, where the miss was $P - y$.

Let us check it three ways: the formula, a finite difference in NumPy, and `jax.grad`.


In [ ]:
def softmax(S, axis=-1):
    E = np.exp(S - S.max(axis=axis, keepdims=True))     # subtract the max: see below
    return E / E.sum(axis=axis, keepdims=True)

def logsumexp(S, axis=-1):
    m = S.max(axis=axis, keepdims=True)
    return (m + np.log(np.exp(S - m).sum(axis=axis, keepdims=True))).squeeze(axis)

def ce_one(s, c):
    return logsumexp(s) - s[c]            # = -log p_c

s_ex = np.array([2.0, 1.0, 0.1]); c_ex = 1
y_onehot = np.eye(3)[c_ex]
formula = softmax(s_ex) - y_onehot
h = 1e-6
fin_diff = np.array([(ce_one(s_ex + h * np.eye(3)[j], c_ex) - ce_one(s_ex - h * np.eye(3)[j], c_ex)) / (2 * h)
                     for j in range(3)])
by_grad = np.array(grad(lambda s: jax.nn.logsumexp(s) - s[c_ex])(jnp.array(s_ex)))
np.set_printoptions(precision=7)
print("scores              :", s_ex, " true class:", c_ex)
print("probabilities p     :", softmax(s_ex))
print("formula p - y       :", formula)
print("finite difference   :", fin_diff)
print("jax.grad            :", by_grad)
np.set_printoptions(precision=4)
print("loss                :", round(float(ce_one(s_ex, c_ex)), 4))


**Reading.** The probabilities are 0.6590, 0.2424, 0.0986. The true class is 1, so $p - y$ is 0.6590, -0.7576, 0.0986. The three methods agree to about 6 digits (float32 `jax.grad` differs from the formula only in the 7th digit). Notice the signs: the true class has a negative entry (its score should go up) and the others are positive (their scores should go down).


### The stable version

The exponential overflows quickly. In float64, $e^{710}$ is already `inf`. The fix uses the fact that softmax does not change if you subtract the same number from every score. Subtract the largest score, and the biggest exponent is $e^0 = 1$.

For the loss, use **log-sum-exp**: $\log\sum_k e^{s_k} = m + \log\sum_k e^{s_k - m}$ with $m = \max_k s_k$.


In [ ]:
s_big = np.array([1000.0, 1001.0, 1002.0])
with np.errstate(all="ignore"):
    naive = np.exp(s_big) / np.exp(s_big).sum()
print("naive softmax of (1000, 1001, 1002):", naive)
print("stable softmax                     :", softmax(s_big))
print("stable log-sum-exp                 :", round(float(logsumexp(s_big)), 4))


**Reading.** The naive version gives `nan` (it computes inf divided by inf). The stable one gives 0.0900, 0.2447, 0.6652: the same as for the scores (0, 1, 2), because only the gaps between scores matter.

This is why libraries want **raw scores**. `nn.CrossEntropyLoss` (K classes) and `nn.BCEWithLogitsLoss` (two classes) apply the stable softmax or sigmoid inside the loss. Never put a softmax or a sigmoid before them.


### Computing corner: axes in the loss

A batch of 4 examples and 3 classes has scores of shape (4, 3): rows are examples (axis 0, the batch axis), columns are classes (axis 1, the class axis).

- Softmax and log-sum-exp go **along the class axis**: `axis=1`. Each row then sums to 1.
- The mean of the loss goes **along the batch axis**: one number per example, then average.
- With the wrong axis the code still runs and gives numbers of the right shape. Only the values are wrong. Always check that the rows sum to 1.


In [ ]:
S_b = np.array([[2.0, 0.0, -1.0], [0.5, 0.5, 0.5], [0.0, 3.0, 1.0], [-1.0, 0.0, 2.0]])
lab_b = np.array([0, 2, 1, 2])
P_ok = softmax(S_b, axis=1)
P_bad = softmax(S_b, axis=0)
print("axis=1 row sums:", P_ok.sum(axis=1))
print("axis=0 row sums:", P_bad.sum(axis=1), " column sums:", P_bad.sum(axis=0))
per_example = logsumexp(S_b, axis=1) - S_b[np.arange(4), lab_b]
print("loss per example:", per_example, " mean:", round(per_example.mean(), 4))

# nn.CrossEntropyLoss takes raw scores
S_t = torch.tensor(S_b); lab_t = torch.tensor(lab_b)
loss_raw = nn.CrossEntropyLoss()(S_t, lab_t).item()
loss_twice = nn.CrossEntropyLoss()(torch.softmax(S_t, dim=1), lab_t).item()
print(f"CrossEntropyLoss on raw scores   : {loss_raw:.4f}")
print(f"CrossEntropyLoss after a softmax : {loss_twice:.4f}   (wrong: softmax applied twice)")


**Reading.** With `axis=1` the rows sum to 1. With `axis=0` it is the columns that sum to 1 and the rows do not (0.783, 0.364, ...). The mean loss is 0.4020, and `nn.CrossEntropyLoss` on raw scores gives the same number (0.4020). If you apply a softmax yourself first, the loss function applies another one inside, and you get 0.7680: it runs, and it is wrong.


**Your turn.** The cell below uses the wrong axis on purpose. Change the line marked `# change this` so that every row of `P2` sums to 1. The cell prints the row sums. The cell also prints `loss2`, the loss of each of the 4 examples for the labels `lab2`.


In [ ]:
S2 = np.array([[1.0, 2.0, 3.0], [3.0, 2.0, 1.0], [0.0, 0.0, 5.0], [1.0, -1.0, 0.0]])
lab2 = np.array([2, 0, 2, 1])
P2 = softmax(S2, axis=0)  # change this
print("row sums:", P2.sum(axis=1))
print("target: all four row sums equal 1")
loss2 = logsumexp(S2, axis=1) - S2[np.arange(4), lab2]   # loss per example
print("loss per example:", loss2)


### Computing corner: dtypes

A **dtype** is the number format of a tensor. Fewer bits means less memory and faster math, but a smaller range and fewer digits.

- `float64`: NumPy's default. 8 bytes per number.
- `float32`: PyTorch's and JAX's default. 4 bytes.
- `float16`: 2 bytes. Used on GPUs. Largest value is only 65,504.
- `bfloat16` (bf16): 2 bytes. Same range as float32 but only about 2 to 3 digits.


In [ ]:
for dt in (torch.float64, torch.float32, torch.float16, torch.bfloat16):
    fi = torch.finfo(dt)
    print(f"{str(dt):15s} bytes {fi.bits // 8}   largest {fi.max:.3g}   spacing near 1: {fi.eps:.1e}")

s16 = np.array([12.0, 11.0, 10.0], dtype=np.float16)
with np.errstate(all="ignore"):
    naive16 = np.exp(s16) / np.exp(s16).sum()
stable16 = softmax(s16)
print("float16 naive softmax of (12, 11, 10):", naive16)
print("float16 stable softmax               :", stable16, stable16.dtype)


**Reading.** In float16 the largest number is 65,504, and $e^{12}$ = 162,755 is bigger, so `np.exp(12)` becomes `inf` and the naive softmax gives `nan`. After the max-subtraction the largest exponent is $e^0 = 1$, and the result is fine: 0.666, 0.245, 0.090. bf16 has the range of float32, so it does not overflow here, but its spacing near 1 is 7.8e-03, about 8 times coarser than float16 (9.8e-04).


**Online lab.** Open the [Next-Token Lab](https://vsokolov.org/labs/next-token-lab.html) next to this section. Try this: find the list of next-word probabilities and check that they add to 1, as a softmax row does.


**Check yourself.**

1. In `softmax(S, axis=1)` for scores of shape (128, 10), what does each row sum to, and what is the shape of the result?
2. Why does `nn.CrossEntropyLoss` want raw scores and not probabilities?
3. Scores are (1000, 1001, 1002). Why is the stable softmax the same as the softmax of (0, 1, 2)?

<details><summary>Answer</summary>

1. Each row sums to 1. The shape stays (128, 10).
2. It computes the softmax and the log together in a stable way (log-sum-exp). If you apply a softmax first, it is applied twice.
3. Subtracting the same number from every score does not change the ratios $e^{s_j} / \sum_k e^{s_k}$.

</details>


In [ ]:
# Inputs for Exercise 12.3
S_ex = np.array([[2.0, 0.5, -1.0], [0.0, 0.0, 0.0], [1.0, 3.0, 0.0], [-1.0, 0.0, 2.0]])
lab_ex = np.array([0, 2, 1, 2])


### Exercise 12.3 (In class)

**Task:** for a batch of 4 examples and 3 classes, compute the probabilities, the loss, and the gradient of the loss with respect to the scores. Use array operations along the class axis (`axis=1`). Use the stable softmax (subtract each row's maximum).

- Inputs: `S_ex` (array of shape (4, 3), raw scores), `lab_ex` (array of 4 class labels, each 0, 1 or 2).
- Define `P_ex` (array of shape (4, 3)): the softmax probabilities, each row sums to 1.
- Define `loss_ex` (a number): the cross-entropy averaged over the 4 examples.
- Define `grad_scores` (array of shape (4, 3)): the gradient of `loss_ex` (the **mean** loss) with respect to the scores. Per example it is $p - y$, and the mean over 4 examples divides by 4: `grad_scores = (P_ex - Y) / 4`.

How: make the one-hot matrix `Y = np.eye(3)[lab_ex]` (shape (4, 3)). `S_ex.max(axis=1, keepdims=True)` keeps shape (4, 1), so it can be subtracted row by row. The true-class probabilities are `P_ex[np.arange(4), lab_ex]` (4 numbers); `loss_ex` is minus the mean of their log (a single number). Then `grad_scores = (P_ex - Y) / 4`. The check compares your whole array with the expected one and with `jax.grad` of the same mean loss.


In [ ]:
# Exercise 12.3: define P_ex, loss_ex, grad_scores
P_ex = None
loss_ex = None
grad_scores = None


In [ ]:
assert P_ex is not None and np.shape(P_ex) == (4, 3), "P_ex must have shape (4, 3)"
assert np.allclose(P_ex.sum(axis=1), 1), "rows of P_ex must sum to 1"
assert np.allclose(P_ex, np.array([[0.7855970346, 0.1752903921, 0.0391125733], [0.3333333333, 0.3333333333, 0.3333333333], [0.1141951994, 0.8437947345, 0.0420100661], [0.0420100661, 0.1141951994, 0.8437947345]]), atol=1e-8), "P_ex does not match: subtract each row maximum, use axis=1"
check("loss_ex", loss_ex, "e9f3f8889d8c51b7")
assert grad_scores is not None and np.shape(grad_scores) == (4, 3), "grad_scores must have shape (4, 3)"
assert np.allclose(grad_scores, np.array([[-0.0536007414, 0.043822598, 0.0097781433], [0.0833333333, 0.0833333333, -0.1666666667], [0.0285487998, -0.0390513164, 0.0105025165], [0.0105025165, 0.0285487998, -0.0390513164]]), atol=1e-8), "grad_scores should be (P_ex - Y) / 4, the gradient of the MEAN loss"
ref = np.array(jax.grad(lambda S: jnp.mean(jax.nn.logsumexp(S, axis=1) - S[jnp.arange(4), lab_ex]))(jnp.array(S_ex)))
assert np.allclose(grad_scores, ref, atol=1e-6), "grad_scores must equal the autodiff gradient of loss_ex"
print("ok  exercise")


## 4. Minibatches: SGD and the memory budget

The circle network used all 200 points at every step (**full batch**). With millions of examples that is too slow and too big for memory. **Minibatch SGD** uses a random subset of $m$ examples per step:

1. Shuffle the data. Cut it into batches of $m$ examples.
2. For each batch, compute the gradient of the average loss on that batch.
3. Take the step $w \leftarrow w - \eta \times \text{gradient}$.

One pass through all the data is an **epoch**. With $n$ examples and batch size $m$, one epoch has $n/m$ steps.

The data below are **simulated**: 6,000 examples, 10 inputs, 3 classes, labels made by a hidden random network plus noise. The network is 10-32-3: a sigmoid hidden layer of 32 units and 3 scores, trained with softmax cross-entropy and plain SGD at $\eta = 0.5$ for 20 epochs.


In [ ]:
rng = np.random.default_rng(12)
n_obs, n_in = 6000, 10
Xs = rng.normal(size=(n_obs, n_in))
T1 = rng.normal(size=(n_in, 8))
T2 = 1.5 * rng.normal(size=(8, 3))
Ys = (np.tanh(Xs @ T1) @ T2 + 0.5 * rng.gumbel(size=(n_obs, 3))).argmax(axis=1)
Xj, Yj = jnp.array(Xs), jnp.array(Ys)
print("class counts:", np.bincount(Ys))

def init_net(key, hidden=32):
    ka, kb = random.split(key)
    return [0.3 * random.normal(ka, (n_in, hidden)), jnp.zeros(hidden),
            0.3 * random.normal(kb, (hidden, 3)), jnp.zeros(3)]

def loss_net(p, x, y):
    z = jax.nn.sigmoid(x @ p[0] + p[1])
    s = z @ p[2] + p[3]
    return jnp.mean(jax.nn.logsumexp(s, axis=1) - s[jnp.arange(len(y)), y])

@jit
def sgd_step(p, x, y, lr):
    g = grad(loss_net)(p, x, y)
    return [a - lr * b for a, b in zip(p, g)]

def train_minibatch(m, lr=0.5, epochs=20, seed=0):
    order = np.random.default_rng(seed)
    p = init_net(random.PRNGKey(0))
    curve, t0 = [float(loss_net(p, Xj, Yj))], time.time()
    for ep in range(epochs):
        idx = order.permutation(n_obs)
        for i in range(0, n_obs, m):
            j = idx[i:i + m]
            p = sgd_step(p, Xj[j], Yj[j], lr)
        curve.append(float(loss_net(p, Xj, Yj)))
    return curve, (time.time() - t0) / epochs

fig, ax = plt.subplots(figsize=(5, 3.5))
results = {}
print(f"starting loss (all three runs): {float(loss_net(init_net(random.PRNGKey(0)), Xj, Yj)):.3f}")
for m, color, style in ((n_obs, "C0", "-"), (128, "C3", "--"), (16, "k", ":")):
    curve, secs = results[m] = train_minibatch(m)
    ax.plot(curve, color=color, ls=style, label=f"batch {m}")
    print(f"batch {m:5d}: {-(-n_obs // m):5d} steps per epoch, loss after 20 epochs {curve[-1]:.3f}, {secs:.3f} s per epoch")
ax.set_xlabel("epoch")
ax.set_ylabel("training loss (average cross-entropy)")
ax.legend(frameon=False)
fig.tight_layout()


**Reading.** All three start at the same loss (1.239). After 20 epochs the full-batch loss is 0.720, batch 128 gives 0.339, and batch 16 gives 0.230. Per epoch, smaller batches make many more updates (1 step for full batch, 47 for 128, 375 for 16), so they learn more per epoch. The number of steps is $n/m$ rounded up: 6000/128 gives 47 steps, and the last batch has 112 examples. The time column shows the price: many small steps in a Python loop take more seconds per epoch (times differ on your machine). On a GPU a tiny batch also leaves the chip mostly idle. The batch size is a trade between steps that are cheap but noisy and steps that are expensive but accurate.


### Is the minibatch gradient noisy but unbiased?

Fix the weights. Compute the full-data gradient once. Then draw 1,000 random minibatches and compute the gradient of each. We follow one entry of the gradient (the first output weight).


In [ ]:
p0 = init_net(random.PRNGKey(0))
g_full = float(grad(loss_net)(p0, Xj, Yj)[2][0, 0])
gdraw = grad(loss_net)
sd_by_m = {}
print(f"full-data gradient entry: {g_full:.4f}")
for m in (16, 128):
    r = np.random.default_rng(5)
    vals = np.array([float(gdraw(p0, Xj[j], Yj[j])[2][0, 0])
                     for j in (r.choice(n_obs, m, replace=False) for _ in range(1000))])
    sd_by_m[m] = vals.std()
    print(f"batch {m:3d}: mean of 1000 minibatch gradients {vals.mean():.4f}, standard deviation {vals.std():.4f}")


**Reading.** The full gradient entry is -0.0579. The average of 1,000 minibatch gradients is very close to it for both sizes: **unbiased**, because a random batch is a fair sample of the data. Single minibatches scatter around it: the standard deviation is 0.0673 for batch 16 and 0.0248 for batch 128, a ratio of 2.72. That is close to $\sqrt{128/16} = 2.83$: the noise falls like one over the square root of the batch size.


### Computing corner: memory in bytes

A tensor takes **number of entries times bytes per entry**. Backpropagation must keep the activations of every layer from the forward pass, because `dA1 = dZ * Z * (1 - Z)` needs `Z`. So memory grows with the batch size $m$.

- A batch of $m$ examples through a layer with $h$ units stores $m \times h$ numbers.
- The numbers below are an **illustrative subtotal**: only some stored arrays. They leave out the weights, their gradients, other intermediate arrays and library workspace, so the real footprint is larger.
- Float32 uses 4 bytes per number. 1 megabyte (MB) is $10^6$ bytes.


In [ ]:
def nbytes(shape, bytes_per_entry):
    return int(np.prod(shape)) * bytes_per_entry

# a batch of 128 images of 28 x 28 pixels, float32
batch_bytes = nbytes((128, 28, 28), 4)
print(f"128 images of 28 x 28 in float32: {batch_bytes:,} bytes = {batch_bytes / 1e6:.3f} MB")

# activations kept for the backward pass in a 784-512-512-10 network, batch of 128
layers = {"input": 784, "hidden 1": 512, "hidden 2": 512, "scores": 10}
total = 0
for name, width in layers.items():
    b = nbytes((128, width), 4)
    total += b
    print(f"  {name:9s} (128, {width:3d}): {b:>9,} bytes")
print(f"subtotal of activations kept, batch 128: {total:,} bytes = {total / 1e6:.2f} MB (weights, gradients and workspace not counted)")
print(f"same network, batch 128 in float16: {total // 2:,} bytes")
weights_70b = 70e9 * 2
print(f"a model with 70 billion weights in float16: {weights_70b / 1e9:.0f} GB (more than one 80 GB GPU holds, before any activations or gradients)")


**Reading.** 128 images of $28 \times 28$ pixels take $128 \times 784 \times 4$ = 401,408 bytes, about 0.40 MB. The stored activations of a small network for a batch of 128 total 0.93 MB (an illustrative subtotal: weights, gradients, other intermediates and workspace come on top). That is nothing for a modern machine. But memory grows in proportion to $m$ and to the width and depth of the network. That is why big models need big memory. And the weights alone of a 70-billion-weight model in float16 take 140 GB, more than one 80 GB GPU can hold, before any activations or gradients. That is the other reason to use many GPUs, which Section 5 mentions.


**Your turn.** How much memory do the activations take for the same network if the batch is 1,024 and the numbers are float32? Edit the line marked `# change this` and run. Target: the printed value is 8 times the batch-128 value above.


In [ ]:
m_try = 128  # change this
mem_try = sum(nbytes((m_try, w), 4) for w in (784, 512, 512, 10))
print(f"batch {m_try}: {mem_try / 1e6:.2f} MB")


**Check yourself.**

1. Why is the minibatch gradient noisy but unbiased?
2. Going from batch 16 to batch 64, by what factor does the noise (standard deviation) fall?
3. A tensor has shape (32, 3, 224, 224) and dtype float16. How many bytes?

<details><summary>Answer</summary>

1. Unbiased: the average over all possible random batches of the same size is the full gradient. Noisy: any single batch is a small sample, so its gradient differs from the full one.
2. By a factor of $\sqrt{64/16} = 2$.
3. $32 \times 3 \times 224 \times 224 \times 2$ = 9,633,792 bytes, about 9.6 MB.

</details>


## 5. Many GPUs: data-parallel SGD as map-reduce

One GPU is not enough for the biggest jobs. The simplest way to use many is **data parallelism**:

- Every GPU holds a full copy of the model.
- Cut each minibatch into shards, one per GPU. Here we use 4 shards.
- **Map:** each GPU computes the gradient on its own shard.
- **Reduce:** average the 4 gradients. On real hardware this is an **all-reduce**: every GPU ends up with the same average.
- Every copy of the model takes the same step, so the copies stay identical.

**Instructor note.** The identity below assumes every shard uses the same model weights and a loss that is an average of per-example terms. Operations that mix the examples of a batch, such as batch normalization, need more care.

We simulate this on one CPU: the "GPUs" are 4 slices of the batch. The claim to test: the average of the 4 shard gradients **equals** the gradient on the full batch. It is the same identity as the summed counts in the experiments topic: a sum (or an average with equal sizes) can be computed in pieces.


In [ ]:
idx_batch = np.arange(128)                       # one minibatch of 128 examples
xb, yb = Xj[idx_batch], Yj[idx_batch]
g_full_batch = grad(loss_net)(p0, xb, yb)

shard_ids = np.array_split(idx_batch, 4)         # 4 shards of 32
shard_grads = [grad(loss_net)(p0, Xj[s], Yj[s]) for s in shard_ids]            # map
g_avg = [sum(g[i] for g in shard_grads) / 4 for i in range(4)]                  # reduce
diffs = [float(jnp.abs(a - b).max()) for a, b in zip(g_avg, g_full_batch)]
print("shard sizes:", [len(s) for s in shard_ids])
print("largest difference per weight array:", [f"{d:.1e}" for d in diffs])
print("largest gradient entry:", f"{max(float(jnp.abs(a).max()) for a in g_full_batch):.3f}")


**Reading.** The largest difference between the averaged shard gradient and the full-batch gradient is 5.2e-08, while gradient entries are about 0.22. The tiny difference is float32 rounding (adding numbers in a different order), not a real gap. In exact arithmetic they are equal. Exercise 12.4 shows it in float64.


In [ ]:
# Training for 50 steps: one copy on batch 128, against 4 shards with averaged gradients.
pa = init_net(random.PRNGKey(0))
pb = init_net(random.PRNGKey(0))
for t in range(50):
    ids = np.random.default_rng(t).choice(n_obs, 128, replace=False)
    pa = sgd_step(pa, Xj[ids], Yj[ids], 0.5)
    sg = [grad(loss_net)(pb, Xj[s], Yj[s]) for s in np.array_split(ids, 4)]
    pb = [w - 0.5 * sum(g[i] for g in sg) / 4 for i, w in enumerate(pb)]
print("largest difference between the two sets of weights after 50 steps:",
      f"{max(float(jnp.abs(a - b).max()) for a, b in zip(pa, pb)):.1e}")
n_params = sum(int(np.prod(a.shape)) for a in pa)
print(f"parameters: {n_params}, gradient message per step: {n_params * 4:,} bytes")
print(f"for a 1-billion-weight model in float32: {1e9 * 4 / 1e9:.0f} GB per step per GPU")


**Reading.** After 50 steps the two networks differ by at most 3.0e-07. Data-parallel training takes the same steps as one big batch. There are two costs:

- **Communication.** Every step each GPU must exchange its gradient. Our small network has 451 weights, so the message is 1,804 bytes. For a model with a billion weights it is 4 GB per step, and the network between GPUs becomes the bottleneck.
- **A bigger effective batch.** With 4 shards of 32 the step uses 128 examples, 4 times more than one GPU alone would. Less noise per step, but the best learning rate often has to change (by hand, or by a rule of thumb that you will meet in the DL course).

**Unequal shards.** If the shards have different sizes, a plain average of the shard gradients is wrong: weight each shard gradient by its size (divide the size-weighted sum by the total number of examples).

**Model parallelism** is the other case: a model too big for one GPU is split across GPUs (each holds some layers), and data parallelism is not enough.


**Recorded on two GPUs.** Two NVIDIA A100 (80 GB) GPUs in one server on GMU's Hopper cluster, 22 October 2026. Each GPU computed the gradient on its own shard, and an all-reduce averaged the two. The largest difference between the averaged gradient and the full-batch gradient was 2.2e-08, which is float32 rounding: the two are the same. Then the all-reduce alone was timed for gradients of different sizes (float32, 4 bytes per number):

| Parameters | Gradient size | Time per all-reduce | Rate |
|---|---|---|---|
| 1 million | 4 MB | 0.099 ms | 40 GB/s |
| 10 million | 40 MB | 0.66 ms | 59 GB/s |
| 100 million | 400 MB | 5.86 ms | 67 GB/s |
| 1 billion | 4,000 MB | 54 ms | 73 GB/s |

For scale, one 8,192 × 8,192 bfloat16 matrix product took 3.7 ms on the same GPU. Averaging the gradient of a 100-million-parameter model (5.9 ms) costs about as much as 1.6 such products, at every step. Communication grows with the size of the model, which is why large training runs send gradients while the next computation is already running.


### Watts and megawatts

Training takes **energy**, and energy has a cost in money and in carbon.

- **Energy = power $\times$ time.** The unit of power is the watt (W): one joule per second. A GPU that draws 700 W for one hour uses $700 \times 3600$ joules = 0.7 kilowatt-hours (kWh).
- **FLOP per joule** is the efficiency of the chip: floating point operations done per joule of energy. It is peak speed (FLOP per second) divided by power (watts).
- Training a model is a fixed number of FLOP. So the energy is the FLOP divided by (FLOP per joule). Below, the energy of a run uses the real speed (40% of peak), while the FLOP per joule at peak uses the peak speed.


**Measured power.** One NVIDIA A100 (80 GB) on GMU's Hopper cluster, 22 October 2026, power read every 0.1 s with `nvidia-smi` while the GPU repeated 8,192 × 8,192 matrix products for 3 seconds. This GPU's power limit is 500 W.

| Work | Power (W) | TFLOP/s | TFLOP per joule | Energy for one product |
|---|---|---|---|---|
| Idle | 65 | | | |
| float32 | 317 | 19 | 0.06 | 18.3 J |
| TF32 | 487 | 140 | 0.29 | 3.8 J |
| bfloat16 | 485 | 289 | 0.60 | 1.8 J |

- TFLOP per joule is TFLOP/s divided by watts: arithmetic done per unit of energy.
- In bfloat16 the GPU draws more power than in float32 (485 W against 317 W) but finishes the same product with 10 times less energy (1.8 J against 18.3 J). This energy saving, together with the speed, is why AI training uses bfloat16.
- The data-center estimate below assumes newer GPUs that draw 700 W each, more than this A100's 500 W limit.


**From one GPU to a data center.** The numbers below are an **assumption-based estimate**, not a measurement. The assumptions are written in the code:

- 700 W per GPU, and every GPU is assumed to draw 700 W all the time.
- 1 MW (megawatt) = 1,000,000 W. 1 MWh (megawatt-hour) = 1,000 kWh = $3.6 \times 10^9$ joules.
- A peak speed of $10^{15}$ FLOP per second per GPU (16-bit arithmetic), and real training uses 40% of it.
- A cluster of 100,000 GPUs.
- Cooling, networking and the rest of the building add a factor of 1.2 to 1.5 on top of the chips.
- A training run of $10^{24}$ FLOP (an assumed size for a very large model).
- A household uses about 10.8 megawatt-hours (MWh) per year.


In [ ]:
gpu_watts = 700
n_gpus = 100_000
peak_flops = 1e15
utilization = 0.40
overhead = (1.2, 1.5)
run_flop = 1e24
household_mwh_per_year = 10.8

chips_mw = gpu_watts * n_gpus / 1e6
building_mw = (chips_mw * overhead[0], chips_mw * overhead[1])
flop_per_joule = peak_flops / gpu_watts
real_flop_per_joule = flop_per_joule * utilization
run_joules = run_flop / real_flop_per_joule
run_mwh = run_joules / 3.6e9                 # 1 MWh = 3.6e9 joules
run_hours = run_flop / (n_gpus * peak_flops * utilization) / 3600
day_mwh = chips_mw * 24
print(f"chips alone: {chips_mw:.0f} MW; with the building: {building_mw[0]:.0f} to {building_mw[1]:.0f} MW")
print(f"one day of the chips: {day_mwh:,.0f} MWh = {day_mwh / household_mwh_per_year:.0f} household-years of electricity")
print(f"peak FLOP per joule: {flop_per_joule:.2e}; at 40% use: {real_flop_per_joule:.2e}")
print(f"a 1e24 FLOP run: {run_mwh:,.0f} MWh for the chips, in {run_hours:.1f} hours on 100,000 GPUs")


**Reading (assumption-based).** 100,000 GPUs at 700 W each draw 70 MW for the chips alone. With cooling and the rest of the building, 84 to 105 MW. In one day the chips use 1,680 MWh, the same energy that 156 households use in one year. The assumed $10^{24}$ FLOP run needs about 486 MWh at the chips and takes about 6.9 hours on the whole cluster. Change any assumption and the answer scales in proportion. Compare these with the measured watts in the table above.


**Check yourself.**

1. Why does the averaged shard gradient equal the full-batch gradient when all shards have the same size?
2. What is communicated in each step of data-parallel SGD?
3. A GPU draws 400 W for 2 hours. How much energy is that in kilowatt-hours?

<details><summary>Answer</summary>

1. The full-batch gradient is an average of per-example gradients. The average of four equal-size averages is the average over all examples.
2. The gradient (one number per weight), so that every copy applies the same update.
3. $0.4\ \text{kW} \times 2\ \text{h} = 0.8$ kWh.

</details>


In [ ]:
# Inputs for Exercise 12.4
rng4 = np.random.default_rng(7)
X_big = rng4.normal(size=(100, 3))
y_big = (rng4.random(100) < sigmoid(X_big @ np.array([1.0, -2.0, 0.5]))).astype(float)
w_cur = np.array([0.2, -0.1, 0.3])

def logistic_grad(w, X, y):
    # average gradient of the log loss for logistic regression: mean of (p - y) x
    return X.T @ (sigmoid(X @ w) - y) / len(y)


### Exercise 12.4 (In class)

**Task:** simulate data-parallel SGD on 100 examples with 4 equal shards of 25 (use `np.array_split`). Show that the average of the 4 shard gradients equals the full gradient.

- Inputs: `X_big` (shape (100, 3)), `y_big` (length 100), `w_cur` (the current weights, length 3), and the function `logistic_grad(w, X, y)` from the inputs cell.
- Define `full_grad` (array of length 3): the gradient on all 100 examples.
- Define `shard_avg_grad` (array of length 3): the average of the 4 shard gradients.
- Define `max_diff` (a number): the largest absolute difference between the two arrays. It should be below $10^{-12}$.


In [ ]:
# Exercise 12.4: define full_grad, shard_avg_grad, max_diff
full_grad = None
shard_avg_grad = None
max_diff = None


In [ ]:
assert shard_avg_grad is not None and full_grad is not None, "define the arrays first"
assert np.allclose(full_grad, logistic_grad(w_cur, X_big, y_big), atol=1e-12), "full_grad is not the full gradient"
real_diff = np.max(np.abs(np.asarray(full_grad) - np.asarray(shard_avg_grad)))
assert real_diff < 1e-12, "the averaged shard gradient should equal the full gradient"
assert abs(max_diff - real_diff) < 1e-15, "max_diff must be the largest absolute difference"
print("ok  exercise")


## 6. The same network in PyTorch, and diagnosing training

The DL course uses PyTorch, so the at-home work stays in PyTorch. We pick the best device that exists: a CUDA GPU, else the Apple GPU (`mps`), else the CPU. Here we run on the CPU. The numbers are the same on any device up to rounding.

The circle network in PyTorch takes about ten lines. The loop has five lines. Each one matches something from Sections 1 to 3.

- `nn.Sequential(nn.Linear(2, 4), nn.Sigmoid(), nn.Linear(4, 1))` is the 2-4-1 network.
- `nn.BCEWithLogitsLoss(reduction="sum")` is the log loss on a raw score, summed as in the JAX version. (PyTorch's default is the mean.)
- `torch.optim.SGD(..., lr=0.02)` is plain gradient descent: $w \leftarrow w - \eta \times \text{gradient}$. Nothing else.


In [ ]:
def pick_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = pick_device()
print("device:", device, "  cuda available:", torch.cuda.is_available())

torch.manual_seed(0)
Xc = torch.tensor(xc, dtype=torch.float32, device=device)
Yc = torch.tensor(yc, dtype=torch.float32, device=device)
net = nn.Sequential(nn.Linear(2, 4), nn.Sigmoid(), nn.Linear(4, 1)).to(device)
loss_fn = nn.BCEWithLogitsLoss(reduction="sum")
opt = torch.optim.SGD(net.parameters(), lr=0.02)

for step in range(1500):
    scores = net(Xc)[:, 0]        # line 1:
    loss = loss_fn(scores, Yc)    # line 2:
    opt.zero_grad()               # line 3:
    loss.backward()               # line 4:
    opt.step()                    # line 5:

with torch.no_grad():
    acc_torch = ((net(Xc)[:, 0] > 0).float() == Yc).float().mean().item()
print(f"accuracy {acc_torch:.3f}, final loss {loss.item():.2f}")


**Reading.** The same kind of network reaches accuracy 1.000 with a final summed loss of 0.53. (PyTorch starts from its own default random weights, so the loss differs a little from the JAX runs.)


**Your turn.** In the loop, the comment after each line number is blank. Write one short phrase for each of the five lines (what does it do?). Then open the answer.

<details><summary>Answer</summary>

1. Forward pass: compute the raw scores for all 200 points.
2. Compute the loss from the scores and the labels.
3. Set all stored gradients to 0. PyTorch adds a new gradient to the old one, so without this the gradients pile up.
4. Backward pass: reverse-mode AD. It fills `.grad` of every weight with the derivative of the loss.
5. The update: $w \leftarrow w - \eta \times \text{.grad}$ for every weight.

</details>

`loss.backward()` is the **same reverse mode** as `jax.grad`. The difference is the style. JAX returns the gradient as a value, from a function you call. PyTorch stores it inside each weight, in `.grad`. Let us check that they agree: put JAX's starting weights into a PyTorch network and compare gradients.


In [ ]:
chk = nn.Sequential(nn.Linear(2, 4), nn.Sigmoid(), nn.Linear(4, 1)).double()
with torch.no_grad():
    chk[0].weight.copy_(torch.tensor(np.array(init_split[0]), dtype=torch.float64).T)   # PyTorch stores (out, in)
    chk[0].bias.copy_(torch.tensor(np.array(init_split[1]), dtype=torch.float64))
    chk[2].weight.copy_(torch.tensor(np.array(init_split[2]), dtype=torch.float64).T)
    chk[2].bias.copy_(torch.tensor(np.array(init_split[3]), dtype=torch.float64))
nn.BCEWithLogitsLoss(reduction="sum")(chk(torch.tensor(xc))[:, 0], torch.tensor(yc)).backward()
jax_g = grad(nll, argnums=(2, 3, 4, 5))(xc, yc, *init_split)
pairs = [(chk[0].weight.grad.T.detach().cpu().numpy(), jax_g[0]), (chk[0].bias.grad.detach().cpu().numpy(), jax_g[1]),
         (chk[2].weight.grad.T.detach().cpu().numpy(), jax_g[2]), (chk[2].bias.grad.detach().cpu().numpy(), jax_g[3])]
print("largest entry of the gradient:", f"{max(np.abs(a).max() for a, _ in pairs):.3f}")
print("largest difference PyTorch .grad vs jax.grad:", f"{max(np.abs(a - np.array(b)).max() for a, b in pairs):.1e}")


**Reading.** The two libraries give the same gradient. The difference (8.0e-06) is JAX's float32 rounding. Note the transposes: PyTorch stores a layer's weight as (outputs, inputs), our JAX arrays are (inputs, outputs).


### Computing corner: NumPy arrays, PyTorch tensors, devices

- `torch.from_numpy(a)` makes a tensor that **shares memory** with the NumPy array `a`. Change one and the other changes.
- `t.detach().cpu().numpy()` goes back to NumPy: `detach()` drops the gradient history (needed if the tensor takes part in autograd), `cpu()` brings it back from a GPU, and `.numpy()` makes the array.
- `t.to(device)` copies a tensor to a GPU. `t.to(torch.float32)` changes the dtype.
- Tensors in one operation must be on the **same device**. A matrix product also needs matching floating dtypes (other operations may promote types). For `nn.CrossEntropyLoss` the scores are floating tensors of shape (B, K) and the labels are integer class numbers, dtype `torch.long`, of shape (B,).
- NumPy arrays default to float64, PyTorch tensors to float32. That is a common source of dtype errors.


In [ ]:
a = np.arange(6, dtype=np.float64).reshape(2, 3)
t = torch.from_numpy(a)
a[0, 0] = 100.0
print("tensor after changing the NumPy array:", t[0].tolist(), " dtype:", t.dtype)

t32 = torch.ones(3, 2)                       # float32 by default
try:
    t @ t32
except RuntimeError as e:
    print("float64 @ float32 stops with:", str(e)[:70])
print("after .to(float32):", (t.to(torch.float32) @ t32).tolist())

print("back to NumPy:", t.detach().cpu().numpy().dtype)
print("device of t:", t.device, "; moved:", t.to(device).device)
print(f"bytes: {t.element_size()} per entry x {t.numel()} entries = {t.element_size() * t.numel()} (t.nbytes = {t.nbytes})")


**Reading.** After `a[0, 0] = 100` the tensor shows 100 in its first entry: the two share memory. Multiplying a float64 tensor by a float32 tensor is an error, and converting with `.to(torch.float32)` fixes it. A float64 tensor of 6 entries uses 48 bytes.


### Diagnosing training

Plain SGD has few knobs, and most problems show up in the loss curve. We reuse the simulated data of Section 4 with the first 2,000 examples for training (full batch) and the last 3,000 for validation. In `train_net` an "epoch" is one full-batch step (the whole training set at once), unlike Section 4, where an epoch is a pass of many minibatch steps. Three prepared runs:

- **Flat:** the loss hardly moves. The learning rate is too small (or the update is missing).
- **Exploding:** the loss jumps around or grows. The learning rate is too large.
- **Overfitting:** the training loss keeps falling, but the validation loss turns around and rises. Use only 60 training examples and a network with 256 hidden units (learning rate 0.1, 2,000 epochs).


In [ ]:
Xt = torch.tensor(Xs, dtype=torch.float32, device=device)
Yt = torch.tensor(Ys, device=device)

def train_net(lr, n_train, hidden, epochs, wd=0.0):
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(n_in, hidden), nn.Sigmoid(), nn.Linear(hidden, 3)).to(device)
    optim = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=wd)
    ce = nn.CrossEntropyLoss()
    tr, va = [], []
    best = (float("inf"), None, None)               # (validation loss, epoch, copy of the weights)
    for e in range(epochs):
        optim.zero_grad()
        l = ce(model(Xt[:n_train]), Yt[:n_train])
        l.backward()
        optim.step()
        with torch.no_grad():                        # training and validation at the same checkpoint
            tr.append(ce(model(Xt[:n_train]), Yt[:n_train]).item())
            v = ce(model(Xt[3000:]), Yt[3000:]).item()
        va.append(v)
        if v < best[0]:
            best = (v, e, copy.deepcopy(model.state_dict()))
    return np.array(tr), np.array(va), model, best

runs = {
    "flat (lr 0.0001)": train_net(1e-4, 2000, 32, 200),
    "healthy (lr 0.5)": train_net(0.5, 2000, 32, 200),
    "exploding (lr 1000)": train_net(1000, 2000, 32, 200),
    "overfitting (60 examples)": train_net(0.1, 60, 256, 2000),
}
fig, axes = plt.subplots(1, 4, figsize=(11, 2.8))
for ax, (name, (tr, va, *_)) in zip(axes, runs.items()):
    ax.plot(tr, color="C0", label="training")
    ax.plot(va, color="C3", ls="--", label="validation")
    ax.set_title(name, fontsize=9)
    ax.set_xlabel("epoch")
    if name.startswith("exploding"):
        ax.set_yscale("log")
axes[0].set_ylabel("cross-entropy")
axes[0].legend(frameon=False, fontsize=8)
fig.tight_layout()
for name, (tr, va, *_) in runs.items():
    print(f"{name:27s} training loss {tr[0]:.3f} -> {tr[-1]:.3f}   validation loss {va[0]:.3f} -> {va[-1]:.3f}")


**Reading.**

- Flat: the training loss goes from 1.129 to 1.126. It barely moves in 200 steps.
- Healthy: it falls from 1.102 to 0.443, and the validation loss follows (0.462).
- Exploding: after the first step the loss is already 791.0, and it ends at 49.1 (log scale in the plot), far above the starting loss of about 1.1.
- Overfitting: the training loss reaches 0.166, but the validation loss, after a minimum of 0.689 at epoch 157, climbs to 1.187. The network fits its 60 training examples ever more closely while getting worse on new ones.


**Your turn: a planted bug.** The training function below has a bug. The loss never changes. Find the bug, fix it, and run again. Target: the training loss after 100 epochs is below 0.8. (Hint: print the first weight before and after one step.)


In [ ]:
def train_buggy(lr=0.5, epochs=100):
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(n_in, 32), nn.Sigmoid(), nn.Linear(32, 3)).to(device)
    optim = torch.optim.SGD(model.parameters(), lr=lr)
    ce = nn.CrossEntropyLoss()
    hist = []
    for e in range(epochs):
        optim.zero_grad()
        l = ce(model(Xt[:2000]), Yt[:2000])
        l.backward()
        # optim.step()      # change this
        hist.append(l.item())
    return hist

hist_bug = train_buggy()
print(f"loss at epoch 0: {hist_bug[0]:.3f}, at epoch 99: {hist_bug[-1]:.3f}  (target: below 0.8)")


The fix is to turn `optim.step()` back on. With the update on, the same run gives:


In [ ]:
hist_fixed = train_net(0.5, 2000, 32, 100)[0]
print(f"with the update on: loss at epoch 0: {hist_fixed[0]:.3f}, at epoch 99: {hist_fixed[-1]:.3f}  (target: below 0.8)")


Another common bug: a softmax or sigmoid applied before `CrossEntropyLoss` or `BCEWithLogitsLoss`. It runs and trains, but the loss is wrong and learning is slow (see Section 3).

### Weight decay and early stopping

- **Weight decay** adds a pull toward 0 to every update: $w \leftarrow w - \eta\,(\text{gradient} + \lambda w)$. It corresponds to the Gaussian prior (the MAP estimate) from the regularization and priors topic. `torch.optim.SGD(..., weight_decay=0.01)` sets $\lambda = 0.01$.
- **Early stopping** keeps the weights from the epoch with the lowest validation loss. Save a copy of `model.state_dict()` (use `copy.deepcopy`, since the dictionary points at the live weights) whenever validation improves, and load it back at the end with `model.load_state_dict(...)`.


In [ ]:
tr0, va0, model0, best0 = runs["overfitting (60 examples)"]
tr_wd, va_wd, model_wd, best_wd = train_net(0.1, 60, 256, 2000, wd=0.01)
best = best0[1]
print(f"no decay:   final validation loss {va0[-1]:.3f}, final training loss {tr0[-1]:.3f}")
print(f"decay 0.01: final validation loss {va_wd[-1]:.3f}, final training loss {tr_wd[-1]:.3f}")
print(f"same rule for both (best epoch on validation): no decay {best0[0]:.3f} at epoch {best0[1]}, decay 0.01 {best_wd[0]:.3f} at epoch {best_wd[1]}")
model0.load_state_dict(best0[2])                    # restore the saved weights
with torch.no_grad():
    restored = nn.CrossEntropyLoss()(model0(Xt[3000:]), Yt[3000:]).item()
print(f"validation loss after restoring the saved weights: {restored:.3f}")
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(va0, color="C3", ls="--", label="validation, no decay")
ax.plot(va_wd, color="C0", label="validation, decay 0.01")
ax.axvline(best, color="0.4", ls=":", label="early stop")
ax.set_xlabel("epoch")
ax.set_ylabel("validation cross-entropy")
ax.legend(frameon=False)
fig.tight_layout()


**Reading.** Weight decay reduces the deterioration in validation loss late in this run. At the final epoch, validation loss is 0.739 with decay and 1.187 without it (training loss 0.295 with decay against 0.166 without). Early stopping saves the weights of epoch 157 and restores them: the restored model has validation loss 0.689. When each run restores its own best validation checkpoint, the losses are 0.689 without decay and 0.689 with decay (equal at the displayed precision, not necessarily exactly equal). In this experiment, weight decay does not show an additional improvement at the reported precision after checkpoint selection. Its usefulness depends on the data, model, penalty and training procedure. Neither tool is free: early stopping needs a validation set, and weight decay needs a chosen $\lambda$. Because the epoch was chosen on the same validation curve, the validation loss at that epoch is optimistic on average. A fresh test score can come out higher or lower, so the test set is used once, at the very end.


**Check yourself.**

1. A run's loss is exactly the same at every epoch. Name two possible causes.
2. The training loss falls to nearly 0 and the validation loss rises. What is this called, and name two remedies.
3. What happens if every sigmoid is applied twice (once in the network, once in the loss)?

<details><summary>Answer</summary>

1. The update line is missing (or commented out), or the learning rate is 0 (or so small it looks like 0).
2. Overfitting. Weight decay, early stopping, more data, a smaller network.
3. The loss still runs but it is the wrong function, with gradients that shrink, so learning is slow or poor. Pass raw scores to the loss.

</details>


## Exit problem (on paper, no code)

1. One weight $w = 1$, one example $x = 2$ with label $y = 1$, the model $p = \sigma(w x)$ and the log loss. Compute the derivative of the loss with respect to $w$, and one gradient-descent update with learning rate 0.5. (Use $\sigma(2) = 0.88$.)
2. A batch of 32 examples goes through a layer with 10 inputs and 5 outputs. Write the shape of the input matrix, the weight matrix, the output matrix, and the three matrices of the backward pass (the gradient with respect to the output, the weights, and the input).
3. Four GPUs each compute the gradient on 8 examples. Why does the average of the 4 gradients give the same step as one GPU with a batch of 32?

<details><summary>Answer</summary>

1. $dL/dw = (p - y)\,x = (0.88 - 1) \times 2 = -0.24$. The update is $w = 1 - 0.5 \times (-0.24) = 1.12$.
2. Input (32, 10), weights (10, 5), output (32, 5). Backward: gradient with respect to the output (32, 5), with respect to the weights = input transposed times output gradient, (10, 32) times (32, 5) gives (10, 5), and with respect to the input = output gradient times weights transposed, (32, 5) times (5, 10) gives (32, 10).
3. The gradient of the batch loss is the average of 32 per-example gradients. The average of four averages over 8 examples each (equal sizes) is the average over all 32.

</details>


## At home

Exercise 12.5 is the graded at-home work: a complete PyTorch training run with plain SGD.


In [ ]:
# Setup for Exercise 12.5 (run it first): loaders, a model builder and evaluate(), on Fashion-MNIST or, if the download fails, on offline synthetic data.
from torch.utils.data import DataLoader, Subset, TensorDataset

USE_SYNTHETIC = False   # set to True to skip the download and use the offline synthetic data

def synthetic_split(n_train=4000, n_val=2000, n_test=2000):
    # SIMULATED data: 10 classes, 784 features (shaped like 28 x 28 images). Each class has its own random
    # centre; an example is its class centre (scaled down) plus noise. The three splits are drawn separately.
    r = np.random.default_rng(0)
    centres = r.normal(size=(10, 784))
    def draw(n):
        y = r.integers(0, 10, size=n)
        x = 0.06 * centres[y] + r.normal(size=(n, 784))
        return TensorDataset(torch.tensor(x, dtype=torch.float32).reshape(-1, 1, 28, 28), torch.tensor(y, dtype=torch.long))
    return draw(n_train), draw(n_val), draw(n_test)

DATASET = "synthetic" if USE_SYNTHETIC else "fashion-mnist"
if DATASET == "fashion-mnist":
    try:
        from torchvision import datasets, transforms
        tf = transforms.ToTensor()
        fm_train = datasets.FashionMNIST("data", train=True, download=True, transform=tf)
        fm_test = datasets.FashionMNIST("data", train=False, download=True, transform=tf)
        perm = torch.randperm(60_000, generator=torch.Generator().manual_seed(0)).tolist()
        tr_idx, va_idx = perm[:4000], perm[4000:6000]       # 4,000 training and 2,000 validation images
        train_set, val_set, test_set = Subset(fm_train, tr_idx), Subset(fm_train, va_idx), fm_test
        train_labels = fm_train.targets[tr_idx].numpy()
        test_labels = fm_test.targets.numpy()
    except Exception as err:
        print(f"Fashion-MNIST download failed ({type(err).__name__}); switching to the synthetic data.")
        DATASET = "synthetic"
if DATASET == "synthetic":
    train_set, val_set, test_set = synthetic_split()
    train_labels = train_set.tensors[1].numpy()
    test_labels = test_set.tensors[1].numpy()

train_loader = DataLoader(train_set, batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(0))
train_eval_loader = DataLoader(train_set, batch_size=1000)
val_loader = DataLoader(val_set, batch_size=1000)
test_loader = DataLoader(test_set, batch_size=1000)

def make_model():
    torch.manual_seed(0)
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 64), nn.ReLU(), nn.Linear(64, 10))

def evaluate(model, loader):
    # average cross-entropy and accuracy of the model on a loader, without changing the weights
    model.eval()
    total, correct, count = 0.0, 0, 0
    with torch.no_grad():
        for xb_, yb_ in loader:
            xb_, yb_ = xb_.to(device), yb_.to(device)
            out = model(xb_)
            total += nn.functional.cross_entropy(out, yb_).item() * len(yb_)
            correct += (out.argmax(dim=1) == yb_).sum().item()
            count += len(yb_)
    model.train()
    return total / count, correct / count
print(f"active dataset: {DATASET}")
print("training examples:", len(train_set), " validation examples:", len(val_set), " test examples:", len(test_set))
print("one example has shape", tuple(train_set[0][0].shape), "and there are", len(np.unique(train_labels)), "classes")


### Exercise 12.5 (At home)

**Task:** train a small network in PyTorch with plain SGD and select a checkpoint by validation loss. You write the forward pass and the loss from the logits, `zero_grad`, `backward` and the optimizer step. You record training and validation curves over 5 epochs, save and restore the checkpoint with the lowest validation loss, and then evaluate the restored model on the test data **once**, next to a majority-class baseline. It runs in about a minute on a laptop CPU.

- The setup cell uses a 4,000 / 2,000 / 10,000 split of Fashion-MNIST (10 classes of clothing images). If the download fails it switches by itself to **simulated** data with the same shapes (784 features, 10 classes), or you can set `USE_SYNTHETIC = True`. The setup prints which dataset is active. Accuracies on the two datasets are not comparable: say which one you used.
- Inputs (from the setup cell): `train_loader`, `train_eval_loader`, `val_loader`, `test_loader`, `make_model()`, `evaluate(model, loader)` (returns average loss and accuracy), `train_labels`, `test_labels`, `device`.
- Fill in every line marked `FILL` in the code cell:
  1. Inspect one batch: `x_shape`, `y_shape` (tuples), `out_shape` (the shape of the scores), `x_dtype`, `y_dtype` (strings such as `"torch.float32"`).
  2. In `train_one_epoch`: forward pass (raw scores, the logits), loss from the logits (`loss_fn12` is `nn.CrossEntropyLoss`, which takes raw scores), zero the gradients, backward, step.
  3. In the epoch loop: append the training and validation losses to `train_curve` and `val_curve`; when validation improves, remember `best_epoch` and a **copy** of the weights in `best_state`.
  4. After the loop: restore the best weights into `model`, and plot both curves with labelled axes and a legend.
- Answer the two short questions after the check cell.

**Order of work (the test set is used once).**

1. Build and debug with `FINAL_EVAL = False` in the final cell.
2. Restart the kernel and Run All once, to train from scratch.
3. Set `FINAL_EVAL = True` and run only the final-evaluation cell, once. It evaluates the restored model and the baseline on the test data.
4. Write the last answer, save, and submit **without restarting again**.


In [ ]:
# Exercise 12.5: fill in every line marked FILL
loss_fn12 = nn.CrossEntropyLoss()
model = make_model().to(device)
opt = torch.optim.SGD(model.parameters(), lr=0.1)
EPOCHS = 5

xb0, yb0 = next(iter(train_loader))
x_shape = None    # FILL: the shape of xb0 as a tuple
y_shape = None    # FILL: the shape of yb0 as a tuple
out_shape = None  # FILL: the shape of model(xb0.to(device)) as a tuple
x_dtype = None    # FILL: str(xb0.dtype)
y_dtype = None    # FILL: str(yb0.dtype)

def train_one_epoch(model, loader, opt):
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        scores = None   # FILL 1: forward pass
        loss = None     # FILL 2: loss from scores and yb
        # FILL 3: zero the gradients
        # FILL 4: backward pass
        # FILL 5: the SGD step

train_curve, val_curve = [], []
best_val, best_epoch, best_state = float("inf"), None, None
for epoch in range(EPOCHS):
    train_one_epoch(model, train_loader, opt)
    tr_loss = evaluate(model, train_eval_loader)[0]
    va_loss = evaluate(model, val_loader)[0]
    # FILL 6: append tr_loss to train_curve and va_loss to val_curve
    # FILL 7: if va_loss < best_val, remember best_val, best_epoch and a copy (copy.deepcopy) of model.state_dict()
# FILL 8: load best_state back into model
# FILL 9: plot train_curve and val_curve (labelled axes, legend)


In [ ]:
assert x_shape == tuple(xb0.shape), "x_shape: the shape of one batch of images, as a tuple"
assert y_shape == tuple(yb0.shape), "y_shape: the shape of one batch of labels, as a tuple"
assert out_shape == tuple(model(xb0.to(device)).shape), "out_shape: the shape of the scores (batch size, number of classes)"
assert x_dtype == str(xb0.dtype) and y_dtype == str(yb0.dtype), "dtypes: write str(tensor.dtype)"
assert len(train_curve) == EPOCHS and len(val_curve) == EPOCHS and EPOCHS >= 3, "record one training and one validation loss per epoch (at least 3 epochs)"
assert train_curve[-1] < train_curve[0], "the training loss should fall: check the five lines of train_one_epoch"
assert best_epoch == int(np.argmin(val_curve)), "best_epoch must be the epoch with the lowest validation loss"
assert abs(evaluate(model, val_loader)[0] - min(val_curve)) < 1e-5, "restore the best weights into model (load_state_dict of the saved copy)"
print("ok  exercise")


**Question 1.** In one or two sentences: why do we clear the gradients (`zero_grad`) before `backward()`?

YOUR ANSWER HERE


**Question 2.** In one or two sentences: why do we choose the checkpoint by validation loss and not by training loss or test accuracy?

YOUR ANSWER HERE


In [ ]:
# Step 3: run this cell ONCE, after the clean Run All. Do not restart afterwards.
FINAL_EVAL = False   # change this to True only for the final run
if FINAL_EVAL:
    test_loss12, test_acc12 = evaluate(model, test_loader)           # the restored best-validation model
    majority = int(np.bincount(train_labels).argmax())                # majority class from the TRAINING labels only
    baseline_acc = float((test_labels == majority).mean())            # accuracy on the same test examples
    print(f"dataset: {DATASET}")
    print(f"restored model: test loss {test_loss12:.3f}, test accuracy {test_acc12:.3f}")
    print(f"baseline (always predict class {majority}): test accuracy {baseline_acc:.3f}")
else:
    print("test evaluation skipped (FINAL_EVAL is False)")


**Question 3.** In one or two sentences, compare the two test accuracies above (restored model and baseline) and say which dataset was active.

YOUR ANSWER HERE


**Submit.** This is the last lab. The sequence is: (a) you built and debugged with `FINAL_EVAL = False`; (b) you ran Restart and Run All once; (c) you set `FINAL_EVAL = True` and ran only the final-evaluation cell, once; (d) now save the notebook and submit it on Canvas by 7 December (the last day of classes) **without restarting again**. Every `ok` line should have printed.
